# Data Ingestion & Vector Stores

This notebook covers the basic building blocks for a simple local retrieval pipeline:

- Document ingestion from PDFs, web pages, and CSVs
- Metadata tagging
- Chunking strategies
- Local Hugging Face embeddings on CPU
- Chroma local persistence
- FAISS in-memory vector search
- Optional retriever tracing in LangSmith

LangChain’s retrieval docs describe the pipeline as modular, with loaders, splitters, embeddings, and vector stores that can be swapped without rewriting the app. The embeddings docs also note that local CPU models can be used for short queries, and the Chroma docs show both in-memory and persistent local usage.


## Learning goals

By the end of this notebook, you should be able to:

1. Load documents from a few common source types.
2. Attach useful metadata to each document.
3. Split text into chunks.
4. Build embeddings locally on CPU.
5. Store embeddings in Chroma with local persistence.
6. Create a temporary in-memory FAISS index.
7. Trace retrieval steps later with LangSmith if needed.


## 1) Install packages

Run this once in a notebook cell:

```bash
pip install -U pypdf pandas beautifulsoup4 requests sentence-transformers langchain langchain-community langchain-text-splitters langchain-huggingface langchain-chroma chromadb faiss-cpu
```

If you later want LangSmith retriever traces, add:

```bash
pip install -U langsmith
```


In [1]:
%pip install -qU pypdf pandas beautifulsoup4 requests sentence-transformers langchain langchain-community langchain-text-splitters langchain-huggingface langchain-chroma chromadb faiss-cpu langsmith


Note: you may need to restart the kernel to use updated packages.


  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-tests 1.1.4 requires pytest<9.0.0,>=7.0.0, but you have pytest 9.0.3 which is incompatible.
mlflow 3.10.1 requires pandas<3, but you have pandas 3.0.3 which is incompatible.
sagemaker 2.245.0 requires numpy==1.26.4, but you have numpy 2.4.6 which is incompatible.
sagemaker 2.245.0 requires packaging<25,>=23.0, but you have packaging 26.0 which is incompatible.
sagemaker-serve 1.5.0 requires sagemaker-core>=2.5.0, but you have sagemaker-core 1.0.77 which is incompatible.
sagemaker-train 1.5.0 requires sagemaker-core>=2.5.0, but you have sagemaker-core 1.0.77 which is incompatible.
streamlit 1.50.0 requires packaging<26,>=20, but you have packaging 26.0 which is incompatible.
streamlit 1.50.0 requires pandas<3,>=1.4.0, but you have pandas 3.0.3 which is incompatible.

[

## 2) What counts as a document?

In LangChain retrieval, document loaders ingest data from external sources and return standardized `Document` objects. That keeps the rest of the pipeline consistent even when the source changes.


In [2]:
from typing import List
from pathlib import Path
from datetime import datetime

import requests
import pandas as pd
from bs4 import BeautifulSoup

from langchain_core.documents import Document


## 3) Simple ingestion helpers

These helpers keep the notebook basic:

- PDF → text extraction
- Web page → readable text
- CSV → rows as small documents

Each document gets metadata so you can filter and inspect sources later.


In [3]:
def load_pdf(path: str) -> List[Document]:
    from pypdf import PdfReader

    reader = PdfReader(path)
    docs: List[Document] = []
    for page_num, page in enumerate(reader.pages):
        text = page.extract_text() or ""
        if text.strip():
            docs.append(
                Document(
                    page_content=text,
                    metadata={
                        "source": path,
                        "type": "pdf",
                        "page": page_num + 1,
                    },
                )
            )
    return docs


def load_web_page(url: str) -> List[Document]:
    response = requests.get(url, timeout=20)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()
    text = " ".join(soup.get_text(" ").split())
    return [
        Document(
            page_content=text,
            metadata={"source": url, "type": "web"},
        )
    ]


def load_csv(path: str) -> List[Document]:
    df = pd.read_csv(path)
    docs: List[Document] = []
    for idx, row in df.iterrows():
        row_text = " | ".join(f"{col}: {row[col]}" for col in df.columns)
        docs.append(
            Document(
                page_content=row_text,
                metadata={
                    "source": path,
                    "type": "csv",
                    "row_index": int(idx),
                },
            )
        )
    return docs


## 4) Metadata tagging

Metadata makes retrieval better because it lets you inspect and filter documents by source, page number, row index, or file type.

A simple metadata pattern looks like this:

- `source`
- `type`
- `page`
- `row_index`
- `created_at`
- `topic`


In [4]:
sample_docs = [
    Document(
        page_content="LLM prompts are stored in the prompt history table.",
        metadata={"source": "notes.txt", "type": "text", "topic": "database"},
    ),
    Document(
        page_content="Vector stores support similarity search over embeddings.",
        metadata={"source": "notes.txt", "type": "text", "topic": "vector"},
    ),
]

sample_docs


[Document(metadata={'source': 'notes.txt', 'type': 'text', 'topic': 'database'}, page_content='LLM prompts are stored in the prompt history table.'),
 Document(metadata={'source': 'notes.txt', 'type': 'text', 'topic': 'vector'}, page_content='Vector stores support similarity search over embeddings.')]

## 5) Chunking strategies

LangChain’s retrieval docs say text splitters break large documents into smaller chunks that can be retrieved individually and fit into a model’s context window. 

For basics, keep this simple:

- Character-based chunking: easy and predictable
- Token-based chunking: better when you care about model context size
- Semantic chunking: useful later when you want smarter boundary selection


In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
)

chunks = text_splitter.split_documents(sample_docs)

print('Chunks:', len(chunks))
for i, chunk in enumerate(chunks, 1):
    print(f'--- Chunk {i} ---')
    print(chunk.page_content)
    print(chunk.metadata)


C:\Users\MadhiarasanM\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Chunks: 2
--- Chunk 1 ---
LLM prompts are stored in the prompt history table.
{'source': 'notes.txt', 'type': 'text', 'topic': 'database'}
--- Chunk 2 ---
Vector stores support similarity search over embeddings.
{'source': 'notes.txt', 'type': 'text', 'topic': 'vector'}


## 6) Hugging Face embeddings on CPU

LangChain’s Hugging Face docs say `HuggingFaceEmbeddings` can run open source embedding models locally. The embeddings docs also note that local CPU models can be used for short queries, while hosted APIs add network latency.


In [6]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2'
)

print('Embeddings ready')


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5084.96it/s]


Embeddings ready


## 7) Chroma local persistence

Chroma can run locally without a server. The LangChain docs show `persist_directory` for saving data across runs, and the underlying Chroma client also supports `PersistentClient(path=...)` for local persistence.


In [7]:
from langchain_chroma import Chroma

chroma_dir = './chroma_data'

chroma_store = Chroma(
    collection_name='basic_docs',
    embedding_function=embeddings,
    persist_directory=chroma_dir,
)

chroma_store.add_documents(chunks)

print('Saved chunks to Chroma at', chroma_dir)


Saved chunks to Chroma at ./chroma_data


In [8]:
results = chroma_store.similarity_search('Where are prompt histories stored?', k=2)

for i, doc in enumerate(results, 1):
    print(f'--- Result {i} ---')
    print(doc.page_content)
    print(doc.metadata)


--- Result 1 ---
LLM prompts are stored in the prompt history table.
{'source': 'notes.txt', 'type': 'text', 'topic': 'database'}
--- Result 2 ---
Vector stores support similarity search over embeddings.
{'type': 'text', 'topic': 'vector', 'source': 'notes.txt'}


## 8) FAISS in-memory vector search

FAISS is a good choice when you want a temporary, fast local index during a notebook session or test run.

Here we build it in memory from the same chunks.


In [9]:
from langchain_community.vectorstores import FAISS

faiss_index = FAISS.from_documents(chunks, embeddings)

faiss_results = faiss_index.similarity_search('What is a vector store?', k=2)

for i, doc in enumerate(faiss_results, 1):
    print(f'--- FAISS Result {i} ---')
    print(doc.page_content)
    print(doc.metadata)


C:\Users\MadhiarasanM\AppData\Local\Temp\ipykernel_34392\1928355265.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


--- FAISS Result 1 ---
Vector stores support similarity search over embeddings.
{'source': 'notes.txt', 'type': 'text', 'topic': 'vector'}
--- FAISS Result 2 ---
LLM prompts are stored in the prompt history table.
{'source': 'notes.txt', 'type': 'text', 'topic': 'database'}


## 9) In-memory vs persistent vector stores

### Use Chroma persistence when:
- you want the index saved across notebook runs
- you are building a local app that needs durable search
- you want a simple directory-based store

### Use FAISS in memory when:
- you only need the index for the current process
- you want very fast temporary semantic search
- you are experimenting or testing


## 10) Document ingestion examples you can adapt

The workflow below is the basic pattern:

1. Load content.
2. Attach metadata.
3. Split text.
4. Embed the chunks.
5. Store them in a vector database.
6. Retrieve relevant chunks later.


In [ ]:
# Example layout for local files
# pdf_docs = load_pdf('docs/sample.pdf')
# web_docs = load_web_page('https://example.com/article')
# csv_docs = load_csv('data/sample.csv')
# all_docs = pdf_docs + web_docs + csv_docs
# chunks = text_splitter.split_documents(all_docs)
print('Loader templates are ready.')


## 11) Optional retriever tracing

LangSmith has dedicated retriever trace rendering so you can inspect retrieved documents and diagnose retrieval issues. The retriever trace page says these traces give document-level visibility into a RAG pipeline. citeturn937802view4

For later use, keep tracing enabled with:

```env
LANGSMITH_TRACING=true
LANGSMITH_API_KEY=...
LANGSMITH_PROJECT=your-project
```


In [ ]:
# Skeleton for later LangSmith tracing
# from langsmith import trace
#
# with trace('retrieval-demo', run_type='retriever', project_name='basic-retrieval'):
#     docs = chroma_store.similarity_search('prompt history', k=3)
#     print(docs)

print('Tracing skeleton ready for later.')


## 12) Keep chunking simple at the start

For a first pass, use character-based splitting with modest overlap. That gives you a predictable baseline before you experiment with token-aware or semantic splitting later.

That is enough for basic indexing, retrieval testing, and notebook demos.


## Key takeaways

- Load documents into a common `Document` format.
- Attach metadata as early as possible.
- Start with simple chunking before trying advanced split logic.
- Use `HuggingFaceEmbeddings` on CPU for local embeddings.
- Use Chroma persistence when you want a saved local index.
- Use FAISS in memory for fast temporary search.
- Add LangSmith retriever tracing later when you want visibility into retrieval steps. 


## References

- Document loaders: https://docs.langchain.com/oss/python/integrations/document_loaders/index
- Retrieval building blocks: https://docs.langchain.com/oss/python/langchain/retrieval#building-blocks
- Vector stores: https://docs.langchain.com/oss/python/integrations/vectorstores/index
- Hugging Face embeddings: https://docs.langchain.com/oss/python/integrations/providers/huggingface#huggingfaceembeddings
- Embeddings latency: https://docs.langchain.com/oss/python/integrations/embeddings/index#latency
- LangSmith retriever traces: https://docs.langchain.com/langsmith/log-retriever-trace
